# 1. Fusionnement des CSV et préparation des labels

## 1.1 Chargement de features_clean.csv et features_dirty.csv dans un même DataFrame :

In [4]:
import pandas as pd

clean_file_path = r"C:\Users\josue\Documents\GitHub\cleansight\features_clean.csv"
dirty_file_path = r"C:\Users\josue\Documents\GitHub\cleansight\features_dirty.csv"

# Load the CSV files
df_clean = pd.read_csv(clean_file_path)
df_dirty = pd.read_csv(dirty_file_path)

df_clean["label"] = 0   # 0 = vide
df_dirty["label"] = 1   # 1 = pleine

df = pd.concat([df_clean, df_dirty], ignore_index=True)

## 1.2 Mélange des lignes et sélectionnez vos colonnes d’entrée (features) 

In [7]:
cols_to_drop = [c for c in ["filename", "label", "histogram"] if c in df.columns]
X = df.drop(columns=cols_to_drop)
y = df["label"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# 2. Entraînement d'un arbre de décision (Decision Tree)

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

# 1) Instanciez
dt = DecisionTreeClassifier(max_depth=4, random_state=42)

# 2) Entraînez
dt.fit(X_train, y_train)

# 3) Évaluez
y_pred = dt.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["vide","pleine"]))

# 4) Affichez l’arbre textuel
tree_rules = export_text(dt, feature_names=list(X.columns))
print(tree_rules)

Accuracy: 0.6666666666666666
              precision    recall  f1-score   support

        vide       0.60      1.00      0.75         6
      pleine       1.00      0.33      0.50         6

    accuracy                           0.67        12
   macro avg       0.80      0.67      0.62        12
weighted avg       0.80      0.67      0.62        12

|--- contour_count <= 2205.00
|   |--- mean_g <= 109.21
|   |   |--- class: 1
|   |--- mean_g >  109.21
|   |   |--- file_size <= 27828.50
|   |   |   |--- mean_b <= 114.51
|   |   |   |   |--- class: 1
|   |   |   |--- mean_b >  114.51
|   |   |   |   |--- class: 0
|   |   |--- file_size >  27828.50
|   |   |   |--- class: 0
|--- contour_count >  2205.00
|   |--- mean_r <= 132.38
|   |   |--- class: 1
|   |--- mean_r >  132.38
|   |   |--- class: 0

